In [ ]:
import json
import sys
from typing import List, Dict, Tuple, Set, Any

---

## 📊 Part 1: Exact Match Evaluation

This module computes the standard **Top-K Accuracy** by directly matching the predicted diagnostic candidates against the exact ground-truth labels. It provides a strict and baseline assessment of the model's precision in rare disease identification.

In [11]:
def evaluate_top_k_accuracy(file_path: str, k_values: List[int] = None) -> Tuple[Dict[int, float], Dict[int, int], int]:
    """
    Computes Top-K accuracy for hierarchical multi-label predictions against ground truth.
    
    Args:
        file_path: Path to the JSONL evaluation dataset.
        k_values: List of K thresholds for evaluation. Defaults to [1, 3, 5, 10, 20].
        
    Returns:
        A tuple containing (accuracies, hit_counts, total_samples).
    """
    if k_values is None:
        k_values = [1, 3, 5, 10, 20]
        
    k_values = sorted(k_values)
    hit_counts = {k: 0 for k in k_values}
    total_samples = 0
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                    
                sample = json.loads(line)
                
                # 1. Parse and format ground truth labels
                raw_labels = sample.get('true_label', [])
                if not isinstance(raw_labels, list):
                    raw_labels = [raw_labels]
                ground_truth = {str(label) for label in raw_labels if label}
                
                # 2. Parse prediction candidates
                predictions = sample.get('predict_code', [])
                if not isinstance(predictions, list):
                    continue
                    
                total_samples += 1
                
                # 3. Evaluate hits for each K threshold
                for k in k_values:
                    top_k_sublists = predictions[:k]
                    
                    # Flatten the top-k nested list into a 1D set
                    top_k_candidates = {
                        str(code) 
                        for sublist in top_k_sublists if isinstance(sublist, list) 
                        for code in sublist if code
                    }
                    
                    # Check for intersection (hit)
                    if ground_truth & top_k_candidates:
                        hit_counts[k] += 1
                        
    except FileNotFoundError:
        print(f"Error: Dataset file not found at {file_path}", file=sys.stderr)
        return {}, {}, 0

    # 4. Calculate final accuracies
    accuracies = {}
    if total_samples > 0:
        for k in k_values:
            accuracies[k] = hit_counts[k] / total_samples
            
    return accuracies, hit_counts, total_samples

def print_benchmark_report(accuracies: Dict[int, float], hit_counts: Dict[int, int], total_samples: int):
    """
    Prints a formatted, academic-style ASCII report for the evaluation metrics.
    """
    if total_samples == 0:
        print("No valid samples were evaluated.")
        return

    print("=" * 55)
    print(f"{'EXACT BENCHMARK EVALUATION REPORT':^55}")
    print("=" * 55)
    print(f" Total Evaluated Samples : {total_samples}")
    print("-" * 55)
    print(f" {'Metric':<15} | {'Accuracy (%)':<15} | {'Hits':<10}")
    print("-" * 55)
    
    for k, acc in accuracies.items():
        metric_name = f"Top-{k}"
        acc_percent = acc * 100
        hits = hit_counts[k]
        print(f" {metric_name:<15} | {acc_percent:>10.2f} %   | {hits:<10}")
        
    print("=" * 55)

if __name__ == '__main__':
    # Define dataset path and evaluate
    dataset_path = "./开源数据/3.RareSeek_Phenotype_data.jsonl"
    
    accuracies, hit_counts, total = evaluate_top_k_accuracy(dataset_path)
    print_benchmark_report(accuracies, hit_counts, total)

           EXACT BENCHMARK EVALUATION REPORT           
 Total Evaluated Samples : 529
-------------------------------------------------------
 Metric          | Accuracy (%)    | Hits      
-------------------------------------------------------
 Top-1           |      48.20 %   | 255       
 Top-3           |      60.11 %   | 318       
 Top-5           |      61.44 %   | 325       
 Top-10          |      62.00 %   | 328       
 Top-20          |      62.00 %   | 328       


---

## 🌳 Part 2: Hierarchical Context Evaluation

This module evaluates the predictions using an ontological tree (e.g., ORPHANET classifications). A prediction is considered a "hit" if it matches the ground-truth label or falls within its immediate **ancestors or descendants**, reflecting a more clinically realistic diagnostic radius.

In [ ]:
def _extract_hierarchy_context(data: List[Dict[str, Any]], target_code: str, path: List[str] = None) -> List[Tuple[List[str], List[str]]]:
    """
    Recursively searches the hierarchy tree to find ancestors and descendants for a specific ORPHA code.
    
    Args:
        data: The nested JSON hierarchy data.
        target_code: The specific ORPHA code (without 'ORPHA:' prefix) to search for.
        path: The accumulated path of ancestor codes during recursion.
        
    Returns:
        A list of tuples containing (ancestors_list, descendants_list) for every match found.
    """
    if path is None:
        path = []

    results = []
    
    for item in data:
        current_code = item.get('OrphaCode')
        if not current_code:
            continue
            
        current_path = path + [current_code]
        
        # If the target node is found
        if current_code == target_code:
            # Get immediate parent (if exists) as ancestor
            ancestors = [current_path[-2]] if len(current_path) >= 2 else []
            descendants = []
            
            # Extract 1st and 2nd level descendants
            for child in item.get('Children', []):
                child_code = child.get('OrphaCode')
                if child_code:
                    descendants.append(child_code)
                for grandchild in child.get('Children', []):
                    grandchild_code = grandchild.get('OrphaCode')
                    if grandchild_code:
                        descendants.append(grandchild_code)
            
            results.append((ancestors, descendants))

        # Continue recursion deeper into the tree
        if 'Children' in item:
            results.extend(_extract_hierarchy_context(item['Children'], target_code, current_path))

    return results

def _get_expanded_targets(orpha_codes: List[str], hierarchy_data: List[Dict[str, Any]]) -> Set[str]:
    """
    Expands a list of ORPHA codes to include their immediate hierarchical context (ancestors & descendants).
    """
    expanded_codes = set()
    
    for full_code in orpha_codes:
        # Strip the prefix to match the hierarchy dataset format
        code_stripped = full_code.replace("ORPHA:", "")
        expanded_codes.add(f"ORPHA:{code_stripped}")
        
        context_info = _extract_hierarchy_context(hierarchy_data, code_stripped)
        
        for ancestors, descendants in context_info:
            for related_code in ancestors + descendants:
                expanded_codes.add(f"ORPHA:{related_code}")
                
    return expanded_codes

def evaluate_hierarchical_accuracy(file_path: str, hierarchy_data: List[Dict[str, Any]], k_values: List[int] = None) -> Tuple[Dict[int, float], Dict[int, int], int]:
    """
    Computes Top-K accuracy where predictions are matched against ground truth labels
    expanded by their ontological hierarchy (ancestors and descendants).
    
    Args:
        file_path: Path to the JSONL evaluation dataset.
        hierarchy_data: The parsed JSON dictionary containing the rare disease classification tree.
        k_values: List of K thresholds for evaluation. Defaults to [1, 3, 5, 10, 20].
        
    Returns:
        A tuple containing (accuracies, hit_counts, total_samples).
    """
    if k_values is None:
        k_values = [1, 3, 5, 10, 20]
        
    k_values = sorted(k_values)
    hit_counts = {k: 0 for k in k_values}
    total_samples = 0

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                sample = json.loads(line)

                # 1. Extract and sanitize ground truth
                raw_gold = sample.get('true_label', [])
                if not isinstance(raw_gold, list):
                    raw_gold = [raw_gold]
                gold_standard = [str(code) for code in raw_gold if code]

                # 2. Expand ORPHA codes with their hierarchy
                orpha_labels = [label for label in gold_standard if label.startswith("ORPHA:")]
                hierarchy_expanded_set = _get_expanded_targets(orpha_labels, hierarchy_data)
                
                # Final valid targets: Ontology-expanded ORPHA codes UNION exact match codes (e.g., OMIM)
                valid_targets = hierarchy_expanded_set | set(gold_standard)

                # 3. Extract predictions
                predictions = sample.get("predict_code", [])
                if not predictions:
                    continue
                    
                total_samples += 1

                # 4. Evaluate hits for each K threshold
                for k in k_values:
                    top_k_predictions = set()
                    
                    for pred_group in predictions[:k]:
                        # Handle nested list format
                        if isinstance(pred_group, list):
                            top_k_predictions.update([str(code) for code in pred_group if code])
                        # Fallback for flat string format
                        elif isinstance(pred_group, str) and pred_group:
                            top_k_predictions.add(pred_group)

                    # Check for intersection (hit) against expanded targets
                    if top_k_predictions & valid_targets:
                        hit_counts[k] += 1
                        
    except FileNotFoundError:
        print(f"Error: Evaluation file not found at {file_path}", file=sys.stderr)
        return {}, {}, 0

    # Calculate final normalized accuracies
    accuracies = {}
    if total_samples > 0:
        for k in k_values:
            accuracies[k] = hit_counts[k] / total_samples

    return accuracies, hit_counts, total_samples

def print_benchmark_report(accuracies: Dict[int, float], hit_counts: Dict[int, int], total_samples: int):
    """
    Prints a formatted, academic-style ASCII report for hierarchical evaluation metrics.
    """
    if total_samples == 0:
        print("No valid samples were evaluated.")
        return

    print("=" * 60)
    print(f"{'HIERARCHICAL BENCHMARK EVALUATION REPORT':^60}")
    print("=" * 60)
    print(f" Total Evaluated Samples : {total_samples}")
    print("-" * 60)
    print(f" {'Metric':<20} | {'Accuracy (%)':<15} | {'Hits':<10}")
    print("-" * 60)
    
    for k, acc in accuracies.items():
        metric_name = f"Top-{k} (Hierarchical)"
        acc_percent = acc * 100
        hits = hit_counts[k]
        print(f" {metric_name:<20} | {acc_percent:>10.2f} %      | {hits:<10}")
        
    print("=" * 60)

if __name__ == '__main__':
    # Define paths
    dataset_path = "./开源数据/3.RareSeek_Phenotype_data.jsonl"
    hierarchy_path = "./Classifications of rare diseases.json"
    
    try:
        with open(hierarchy_path, 'r', encoding='utf-8') as f:
            hierarchy_data = json.load(f)
            
        accuracies, hit_counts, total = evaluate_hierarchical_accuracy(dataset_path, hierarchy_data)
        print_benchmark_report(accuracies, hit_counts, total)

    except FileNotFoundError:
        print(f"Error: Hierarchy JSON file not found at {hierarchy_path}", file=sys.stderr)
    except json.JSONDecodeError:
        print(f"Error: Failed to parse hierarchy JSON file.", file=sys.stderr)

          HIERARCHICAL BENCHMARK EVALUATION REPORT          
 Total Evaluated Samples : 528
------------------------------------------------------------
 Metric               | Accuracy (%)    | Hits      
------------------------------------------------------------
 Top-1 (Hierarchical) |      50.38 %      | 266       
 Top-3 (Hierarchical) |      63.26 %      | 334       
 Top-5 (Hierarchical) |      65.15 %      | 344       
 Top-10 (Hierarchical) |      65.72 %      | 347       
 Top-20 (Hierarchical) |      66.67 %      | 352       
